In [1]:
project_root = "/home/krzysztof/studia/magisterka/time-series-invariance"

In [4]:
import os
subdir = ""
os.chdir(os.path.join(project_root, subdir))

import numpy as np

from src.data_generation.simple_data_generation import shift_time_series, shrunk_time_series
from notebooks.evaluations.helpers import (
        apply_embedding_function_shrunk,
        apply_embedding_function_shifts,
        load_dataset,
        load_distortions,
    )

In [5]:
data_dir = "data/datasets"
dataset_names = [name[:-4] for name in os.listdir(data_dir)]
dataset_names

['CBF',
 'OSULeaf',
 'Adiac',
 'Fish',
 'MedicalImages',
 'Wafer',
 'SwedishLeaf',
 'FacesUCR',
 'SyntheticControl',
 'FaceFour',
 'Trace',
 'Beef',
 'GunPoint',
 'TwoPatterns',
 'Coffee',
 'OliveOil',
 'ECG200']

In [7]:
subdir = "ts2vec"
os.chdir(os.path.join(project_root, subdir))

from ts2vec import TS2Vec

print("Model loaded successfully!")

subdir = ""
os.chdir(os.path.join(project_root, subdir))

Model loaded successfully!


In [ ]:
def ts2vec_embedding_function(a: np.ndarray, model) -> np.ndarray:
    if a.ndim == 1:
        a = a[None, :, None]
    if a.ndim == 2:
        a = a[:, :, None]
    v = model.encode(a, encoding_window='full_series')
    return v.reshape(-1)

In [10]:
import re
pattern = re.compile(r"(.+)__test_run_(\d{8})_(\d{6})")

model_paths = {}
model_dir = "ts2vec/training"
for name in os.listdir(model_dir):
    match = pattern.match(name)
    if match:
        dataset_name = match.group(1)
        date = float(match.group(2)) + float(match.group(3))/1e6
        if date > model_paths.get(dataset_name, {}).get("date", 0):
            model_paths[dataset_name] = {
                "date": date,
                "path": os.path.join(model_dir, name, "model.pkl")
            }
model_paths

{'TwoPatterns': {'date': 20260316.121011,
  'path': 'ts2vec/training/TwoPatterns__test_run_20260316_121011/model.pkl'},
 'Beef': {'date': 20260316.120934,
  'path': 'ts2vec/training/Beef__test_run_20260316_120934/model.pkl'},
 'GunPoint': {'date': 20260316.120947,
  'path': 'ts2vec/training/GunPoint__test_run_20260316_120947/model.pkl'},
 'CBF': {'date': 20260316.120325,
  'path': 'ts2vec/training/CBF__test_run_20260316_120325/model.pkl'},
 'Adiac': {'date': 20260316.120555,
  'path': 'ts2vec/training/Adiac__test_run_20260316_120555/model.pkl'},
 'OSULeaf': {'date': 20260316.120349,
  'path': 'ts2vec/training/OSULeaf__test_run_20260316_120349/model.pkl'},
 'ECG200': {'date': 20260316.1211,
  'path': 'ts2vec/training/ECG200__test_run_20260316_121100/model.pkl'},
 'Trace': {'date': 20260316.120911,
  'path': 'ts2vec/training/Trace__test_run_20260316_120911/model.pkl'},
 'Coffee': {'date': 20260316.121035,
  'path': 'ts2vec/training/Coffee__test_run_20260316_121035/model.pkl'},
 'OliveOil

In [21]:
distortion_type = "shrink"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    print(f"data shape: {data.shape}")
    random_shrinks = load_distortions(dataset_name, distortion_type)

    model_path = model_paths[dataset_name]["path"]
    model = TS2Vec(input_dims=1, device='cpu', output_dims=320)
    model.load(model_path)

    
    embeddings = apply_embedding_function_shrunk(data, lambda x: ts2vec_embedding_function(x, model), random_shrinks)
    print(f"embeddings shape: {embeddings.shape}")
    with open(f"notebooks/evaluations/embeddings/ts2vec/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)


data shape: (500, 128)
embeddings shape: (500, 6, 320)
data shape: (442, 427)
embeddings shape: (442, 6, 320)
data shape: (500, 176)
embeddings shape: (500, 6, 320)
data shape: (350, 463)
embeddings shape: (350, 6, 320)
data shape: (500, 99)
embeddings shape: (500, 6, 320)
data shape: (500, 152)
embeddings shape: (500, 6, 320)
data shape: (500, 128)
embeddings shape: (500, 6, 320)
data shape: (500, 131)
embeddings shape: (500, 6, 320)
data shape: (500, 60)
embeddings shape: (500, 6, 320)
data shape: (112, 350)
embeddings shape: (112, 6, 320)
data shape: (200, 275)
embeddings shape: (200, 6, 320)
data shape: (60, 470)
embeddings shape: (60, 6, 320)
data shape: (200, 150)
embeddings shape: (200, 6, 320)
data shape: (500, 128)
embeddings shape: (500, 6, 320)
data shape: (56, 286)
embeddings shape: (56, 6, 320)
data shape: (60, 570)
embeddings shape: (60, 6, 320)
data shape: (200, 96)
embeddings shape: (200, 6, 320)


In [23]:
distortion_type = "shift"
for dataset_name in dataset_names:
    data = load_dataset(dataset_name)
    data = data[:, :, None]
    print(f"data shape: {data.shape}")
    random_shifts = load_distortions(dataset_name, distortion_type)

    model_path = model_paths[dataset_name]["path"]
    model = TS2Vec(input_dims=1, device='cpu', output_dims=320)
    model.load(model_path)

    
    embeddings = apply_embedding_function_shifts(data, lambda x: ts2vec_embedding_function(x, model), random_shifts)
    print(f"embeddings shape: {embeddings.shape}")
    with open(f"notebooks/evaluations/embeddings/ts2vec/{distortion_type}/{dataset_name}.npy", "wb") as f:
        np.save(f, embeddings)

data shape: (500, 128, 1)


AssertionError: 